# Does the trained decision model read the criteria?

Qwen3.5-4B decision model at its best step (844, val MAP 60.4 / 59.5; FINDINGS entry 10), scored on the same 27,870
val records (17 CLEF TAR 2019 Intervention training reviews) with **only `state.review.criteria` changed**:

| Variant | Criteria in the request |
|---|---|
| original | the review's own full-text criteria (scores saved during training) |
| other-review | the criteria of another val review (random derangement, seed 0; `swap.json`) |
| no-criteria | empty string |

Scores: [scripts/criteria_test.py](../../scripts/criteria_test.py), which first checks that the reloaded checkpoint
reproduces the saved scores. Labels: CLEF TAR abstract-level (17 reviews) and content-level (16 reviews), the true
labels of each record's own review in every variant. Ranking per review by P(include), ties by record id.

**Reading:** if the model reads the criteria, other-review must lose much of the MAP over random order, and the
ranking within a review must change (low rank correlation with the original). If it barely changes, the model ranks
by what includes generally look like.

In [ ]:
import json, sys
from pathlib import Path
import numpy as np
import pandas as pd

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "README.md").exists())
sys.path.insert(0, str(ROOT / "src"))
from screening import metrics

SOURCE = "review-criteria__record-title+abstract__label-abstract_else_final__train-neg1"
RUN = ROOT / "runs" / f"Qwen3.5-4B__{SOURCE}"
STEP = 844
OUT = RUN / "criteria_test" / f"step-{STEP}"
JEV_SCORES = ROOT / "results" / f"requests__{SOURCE.removesuffix('__train-neg1')}__val" / "scores.jsonl"

labels = pd.read_json(JEV_SCORES, lines=True)[["review_id", "record_id", "label_abstract", "label_final"]]
scores = {"original": pd.read_json(RUN / "val_scores" / f"step-{STEP:06d}.jsonl", lines=True)[["review_id", "record_id", "p"]]}
for v in ("other-review", "no-criteria"):
    if (OUT / f"{v}.jsonl").exists():
        scores[v] = pd.read_json(OUT / f"{v}.jsonl", lines=True)
for v, s in scores.items():
    assert len(s) == len(labels), f"{v}: {len(s)} records, expected {len(labels)}"
print("variants:", list(scores), "| swap:", json.loads((OUT / "swap.json").read_text())["criteria_of"])

## Ranking quality (macro over reviews)

In [ ]:
def per_review(df, level):
    rows = {}
    for rid, g in df.merge(labels, on=["review_id", "record_id"]).groupby("review_id"):
        m = metrics(g.sort_values(["p", "record_id"], ascending=[False, True])[f"label_{level}"].astype(int).to_numpy())
        if m:
            rows[rid] = m
    return pd.DataFrame(rows).T * 100

rng = np.random.default_rng(0)
def diff(a, b):
    d = (a - b).dropna().to_numpy()
    lo, hi = np.percentile(rng.choice(d, (5000, len(d))).mean(1), [2.5, 97.5])
    return f"{d.mean():+.1f} [{lo:+.1f}, {hi:+.1f}] {(d > 0).sum()}/{(d < 0).sum()}"

R = {(v, level): per_review(s, level) for v, s in scores.items() for level in ("abstract", "final")}
summary = pd.DataFrame([{"variant": v, "labels": level, **R[v, level][["MAP", "R@10%", "WSS@95%", "TNR@95%"]].mean().round(1)}
                        for (v, level) in R]).set_index(["labels", "variant"]).sort_index()
summary

In [ ]:
# Per review: variant − original, 95% bootstrap CI over reviews, reviews better / worse
pd.DataFrame([{"labels": level, "variant": v, "ΔMAP": diff(R[v, level]["MAP"], R["original", level]["MAP"]),
               "ΔWSS@95%": diff(R[v, level]["WSS@95%"], R["original", level]["WSS@95%"])}
              for (v, level) in R if v != "original"])

## Does the ranking within a review change?

Spearman correlation between the original scores and the variant's scores, per review. Near 1: the criteria do not
change which records come first.

In [ ]:
rho = {}
for v in scores:
    if v == "original":
        continue
    m = scores["original"].merge(scores[v], on=["review_id", "record_id"], suffixes=("", "_v"))
    rho[v] = m.groupby("review_id").apply(lambda g: g["p"].rank().corr(g["p_v"].rank()))   # Spearman
rho = pd.DataFrame(rho)
display(rho.describe().loc[["mean", "min", "50%", "max"]].round(2))

In [ ]:
# Per review, abstract-level MAP: own criteria, other review's criteria (and whose), no criteria
swap = json.loads((OUT / "swap.json").read_text())["criteria_of"]
level = "abstract"
table = pd.DataFrame({"includes": labels[labels[f"label_{level}"] == 1].groupby("review_id").size(),
                      **{v: R[v, level]["MAP"] for v in scores},
                      "criteria from": pd.Series(swap)}).join(rho.add_prefix("ρ "))
table.round(2).sort_values("original", ascending=False)